In [44]:
from datasets import load_dataset, Dataset
import os, json, time, random, pathlib
import requests
import numpy as np
import pandas as pd
import torch
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator



In [45]:
os.environ["HF_TOKEN"] = input()

In [46]:
dataset = load_dataset("pameydorke/arcanum-cross-platform-queries-synthetic")

In [47]:
traind_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()
full_df = pd.concat([traind_df, test_df])

unique_docs_df = pd.concat([traind_df, test_df]).drop_duplicates(subset=['document_text'])
unique_docs_df = unique_docs_df.reset_index()
unique_docs_df = unique_docs_df.drop(columns=['index', 'query', 'query_type', 'difficulty'])

unique_docs_ds = Dataset.from_pandas(unique_docs_df)

unique_docs_ds

Dataset({
    features: ['document_title', 'document_text', 'document_link'],
    num_rows: 1153
})

In [48]:
DATA_URL = "https://raw.githubusercontent.com/ankorn/arcanumsearch/refs/heads/main/data/retriever_test_ds.json"
test_ds = requests.get(DATA_URL, timeout=30).json()

In [49]:
FT_MODEL_ID   = "pameydorke/arcanum-cross-platform-retriever"
BASE_MODEL_ID = "BAAI/bge-m3"

In [50]:
base_model = SentenceTransformer('BAAI/bge-m3')
ft_model = SentenceTransformer('pameydorke/arcanum-cross-platform-retriever')

base_model.eval()
ft_model.eval()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)

In [51]:
queries = {}
corpus = {}
relevant_docs = {}
url2text = {}

unique_docs = {}
for idx, example in enumerate(unique_docs_ds):
    doc_link = example["document_link"]
    doc_text = example["document_text"]
    url2text[doc_link] = doc_text

    if doc_text not in unique_docs:
        doc_id = f"doc_{len(unique_docs)}"
        unique_docs[doc_text] = doc_id
        corpus[doc_id] = doc_text

skipped_counter = 0
for idx, example in enumerate(test_ds):
    query = example['query']
    url = example['url']

    if 'https://www.reddit.com' in url:
        url = url.replace('https://www.reddit.com', 'https://reddit.com')

    if url not in url2text:
        skipped_counter += 1
        continue

    doc_text = url2text[url]

    if doc_text not in unique_docs:
        skipped_counter += 1
        continue

    doc_id = unique_docs[doc_text]

    query_id = f"query_{idx}"
    queries[query_id] = query
    relevant_docs[query_id] = {doc_id}

print("Skipped:", skipped_counter)

Skipped: 0


In [52]:
base_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="arcanum-retriever-test",
)
ft_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="arcanum-retriever-test",
)

base_results = base_evaluator(base_model)
ft_results = ft_evaluator(ft_model)

In [53]:
base_results

{'arcanum-retriever-test_cosine_accuracy@1': 0.6515151515151515,
 'arcanum-retriever-test_cosine_accuracy@3': 0.7777777777777778,
 'arcanum-retriever-test_cosine_accuracy@5': 0.8383838383838383,
 'arcanum-retriever-test_cosine_accuracy@10': 0.8585858585858586,
 'arcanum-retriever-test_cosine_precision@1': 0.6515151515151515,
 'arcanum-retriever-test_cosine_precision@3': 0.25925925925925924,
 'arcanum-retriever-test_cosine_precision@5': 0.16767676767676762,
 'arcanum-retriever-test_cosine_precision@10': 0.08585858585858584,
 'arcanum-retriever-test_cosine_recall@1': 0.6515151515151515,
 'arcanum-retriever-test_cosine_recall@3': 0.7777777777777778,
 'arcanum-retriever-test_cosine_recall@5': 0.8383838383838383,
 'arcanum-retriever-test_cosine_recall@10': 0.8585858585858586,
 'arcanum-retriever-test_cosine_ndcg@10': 0.7588491421077267,
 'arcanum-retriever-test_cosine_mrr@10': 0.7262806637806639,
 'arcanum-retriever-test_cosine_map@100': 0.7310997401738089}

In [54]:
ft_results

{'arcanum-retriever-test_cosine_accuracy@1': 0.6111111111111112,
 'arcanum-retriever-test_cosine_accuracy@3': 0.7626262626262627,
 'arcanum-retriever-test_cosine_accuracy@5': 0.8232323232323232,
 'arcanum-retriever-test_cosine_accuracy@10': 0.8737373737373737,
 'arcanum-retriever-test_cosine_precision@1': 0.6111111111111112,
 'arcanum-retriever-test_cosine_precision@3': 0.25420875420875416,
 'arcanum-retriever-test_cosine_precision@5': 0.16464646464646462,
 'arcanum-retriever-test_cosine_precision@10': 0.08737373737373735,
 'arcanum-retriever-test_cosine_recall@1': 0.6111111111111112,
 'arcanum-retriever-test_cosine_recall@3': 0.7626262626262627,
 'arcanum-retriever-test_cosine_recall@5': 0.8232323232323232,
 'arcanum-retriever-test_cosine_recall@10': 0.8737373737373737,
 'arcanum-retriever-test_cosine_ndcg@10': 0.7449045807795477,
 'arcanum-retriever-test_cosine_mrr@10': 0.7034992784992785,
 'arcanum-retriever-test_cosine_map@100': 0.7088135180259474}

In [55]:
import math
import torch
import numpy as np
import pandas as pd
from sentence_transformers import util


def encode_queries_and_corpus(model, queries, corpus, batch_size=64):
    qids = list(queries.keys())
    cids = list(corpus.keys())

    q_texts = [queries[q] for q in qids]
    c_texts = [corpus[c] for c in cids]

    q_emb = model.encode(
        q_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    c_emb = model.encode(
        c_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return qids, cids, q_emb, c_emb

In [56]:
def per_query_metrics(model, queries, corpus, relevant_docs, k_values=(1, 5, 10)):
    qids, cids, q_emb, c_emb = encode_queries_and_corpus(model, queries, corpus)

    # (num_queries, num_corpus)
    sim = util.cos_sim(q_emb, c_emb)

    results = {}

    for i, qid in enumerate(qids):
        scores = sim[i]
        ranked_idx = torch.argsort(scores, descending=True).tolist()
        ranked_cids = [cids[j] for j in ranked_idx]

        rel = relevant_docs.get(qid, set())
        metrics = {}

        for k in k_values:
            topk = ranked_cids[:k]

            # recall@k
            hits = sum(1 for d in topk if d in rel)
            metrics[f"recall@{k}"] = hits / len(rel) if rel else 0.0

            # ndcg@k
            dcg = 0.0
            for rank, d in enumerate(topk, start=1):
                if d in rel:
                    dcg += 1.0 / math.log2(rank + 1)

            idcg = sum(
                1.0 / math.log2(r + 1)
                for r in range(1, min(len(rel), k) + 1)
            )
            metrics[f"ndcg@{k}"] = dcg / idcg if idcg > 0 else 0.0

        # MRR
        rr = 0.0
        for rank, d in enumerate(ranked_cids, start=1):
            if d in rel:
                rr = 1.0 / rank
                break
        metrics["mrr"] = rr

        results[qid] = metrics

    return results

In [57]:
base_per_q = per_query_metrics(base_model, queries, corpus, relevant_docs)

In [58]:
ft_per_q   = per_query_metrics(ft_model,   queries, corpus, relevant_docs)

In [66]:
metric = "recall@5"

base = np.array([base_per_q[q][metric] for q in queries])
ft   = np.array([ft_per_q[q][metric]   for q in queries])
delta = ft - base

print("base mean:", base.mean())
print("ft mean:  ", ft.mean())
print("mean delta:", delta.mean())
print("improved:", (delta > 0).sum())
print("worse:   ", (delta < 0).sum())
print("same:    ", (delta == 0).sum())

rng = np.random.default_rng(0)
n = len(delta)

boot = np.array([
    delta[rng.integers(0, n, n)].mean()
    for _ in range(10_000)
])

lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"mean delta = {delta.mean():.4f}", "\n")
print(f"95% CI     = [{lo:.4f}, {hi:.4f}]")

base mean: 0.8383838383838383
ft mean:   0.8232323232323232
mean delta: -0.015151515151515152
improved: 3
worse:    6
same:     189
mean delta = -0.0152 

95% CI     = [-0.0455, 0.0152]


In [65]:
metric = "ndcg@10"

base = np.array([base_per_q[q][metric] for q in queries])
ft   = np.array([ft_per_q[q][metric]   for q in queries])
delta = ft - base

print("base mean:", base.mean())
print("ft mean:  ", ft.mean())
print("mean delta:", delta.mean())
print("improved:", (delta > 0).sum())
print("worse:   ", (delta < 0).sum())
print("same:    ", (delta == 0).sum())

rng = np.random.default_rng(0)
n = len(delta)

boot = np.array([
    delta[rng.integers(0, n, n)].mean()
    for _ in range(10_000)
])

lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"mean delta = {delta.mean():.4f}", "\n")
print(f"95% CI     = [{lo:.4f}, {hi:.4f}]")

base mean: 0.7588491421077267
ft mean:   0.7449045807795477
mean delta: -0.013944561328179008
improved: 26
worse:    29
same:     143
mean delta = -0.0139 

95% CI     = [-0.0445, 0.0146]


In [61]:
rng = np.random.default_rng(0)
n = len(delta)

boot = np.array([
    delta[rng.integers(0, n, n)].mean()
    for _ in range(10_000)
])

lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"mean delta = {delta.mean():.4f}")
print(f"95% CI     = [{lo:.4f}, {hi:.4f}]")

mean delta = -0.0139
95% CI     = [-0.0445, 0.0146]


In [62]:
rows = []
for qid in queries:
    row = {"qid": qid, "query": queries[qid]}
    for k, v in base_per_q[qid].items():
        row[f"base_{k}"] = v
    for k, v in ft_per_q[qid].items():
        row[f"ft_{k}"] = v
    rows.append(row)

df = pd.DataFrame(rows)
df["delta_ndcg@10"] = df["ft_ndcg@10"] - df["base_ndcg@10"]
df = df.sort_values("delta_ndcg@10")

# Queries where fine-tuning hurt most
print(df[["qid", "query", "base_ndcg@10", "ft_ndcg@10", "delta_ndcg@10"]].head(20))

           qid                                            query  base_ndcg@10  \
194  query_194          steam install UAP after changing config      1.000000   
150  query_150                                mod/patch inquiry      1.000000   
157  query_157     UAP install different version mid-playthrouh      1.000000   
180  query_180                                arcanum locations      1.000000   
33    query_33                                 build calculator      1.000000   
32    query_32                           character build planer      1.000000   
191  query_191                         GOG version on modern PC      1.000000   
188  query_188       return sold quest item by prototype id UAP      1.000000   
101  query_101           Tarnat guards attack after killing npc      1.000000   
154  query_154                                  Melic Bangerton      1.000000   
10    query_10                 Arcane Sword dungeon coordinates      1.000000   
9      query_9  can't play g

In [71]:
# Build a list of (qid, base, ft, delta) for recall@5
rows = []
for qid in queries:
    b = base_per_q[qid]["recall@5"]
    f = ft_per_q[qid]["recall@5"]
    rows.append((qid, b, f, f - b))

# Keep only the ones that got worse
worse = [r for r in rows if r[3] < 0]
worse.sort(key=lambda r: r[3])  # most negative first

print(f"Total worse on recall@5: {len(worse)}")
print("=" * 80)

for qid, b, f, d in worse:
    # print(f"qid:     {qid}")
    print(f"query:   {queries[qid]}")
    # print(f"base:    {b:.3f}   ft: {f:.3f}   delta: {d:+.3f}")

    rel_docs = relevant_docs[qid]
    # print(f"relevant doc ids: {rel_docs}")

    # Show the text of the relevant docs (truncated)
    for doc_id in rel_docs:
        text = corpus[doc_id]
        # print(f"  [{doc_id}] {text[:300]}...")

    # If you want to see where the fine-tuned model ranked the gold doc,
    # you need the ranked list. See helper below.
    print("-" * 80)

Total worse on recall@5: 6
query:   adventure teammate with a elefant gun
--------------------------------------------------------------------------------
query:   bow mastr trainee find
--------------------------------------------------------------------------------
query:   mod/patch inquiry
--------------------------------------------------------------------------------
query:   UAP install different version mid-playthrouh
--------------------------------------------------------------------------------
query:   arcanum locations
--------------------------------------------------------------------------------
query:   steam install UAP after changing config
--------------------------------------------------------------------------------
